# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Qwen estructurado

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

El backbone se documenta mediante el informe Qwen3 [1] y la tarjeta exacta de `Qwen/Qwen3-0.6B-Base` [2]. La separación entre compuerta y daños toma como antecedentes la clasificación jerárquica [3] y multietiqueta [4]. La mejora reutiliza como inicialización entrenable el adaptador LoRA verificable de 03_05 [5], reinicia optimizador y compara una sola época con penalizaciones 0, 0,02 y 0,05. La penalización de conflicto y su regla de selección son locales y se deciden exclusivamente en validation.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab` y asigne una **NVIDIA A100 de 40 GB** para Qwen. El notebook permanece local; Drive transporta solo versiones inmutables del bundle. La celda detecta si falta el release exacto: en ese único caso lo obtiene desde GitHub —o mediante `local_upload`—, verifica todos sus SHA-256 y lo publica de forma atómica. Después promueve la copia activa cuando sea necesario; ya no requiere ejecutar `02_00` previamente. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [6]. La integridad del bundle se comprueba con SHA-256 [7]. No sincronice cachés de modelos ni entrene directamente sobre Drive; los cuadernos de entrenamiento Qwen copian cada checkpoint terminado mediante un TAR atómico y reanudable.

In [1]:
# Backend reproducible: local o Google Colab desde VS Code
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
import uuid
import zipfile

COLAB_NOTEBOOK_ID = "03_06"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = False
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_AUTO_PUBLISH_MISSING_BUNDLE = True
COLAB_BUNDLE_SOURCE = "github"  # "github" o "local_upload"
COLAB_GITHUB_REPOSITORY = "lkoc/Trabajo_PLN-MIA-Grupo4"
COLAB_GITHUB_REF = "main"
COLAB_GITHUB_BUNDLE_PATH = "resultados/colab_bundle"
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "b2030f898ec03e9c7dc6b938de7453bef027c7a19b07249246595d8f4b1629cb"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "862cada62e73b099a817d15748a06cfcc19a55a004939ef7aa3cd88576a9b161"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _verify_expected_bundle(bundle_dir, expected_bundle_id=COLAB_NOTEBOOK_BUILD_BUNDLE_ID):
    manifest_path = Path(bundle_dir) / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta {manifest_path}")
    manifest = _read_manifest(manifest_path)
    computed_bundle_id = _bundle_id_for_manifest(manifest)
    if manifest.get("bundle_id") != computed_bundle_id:
        raise ValueError("bundle_manifest.json no contiene una identidad válida")
    if computed_bundle_id != expected_bundle_id:
        raise ValueError(
            f"Bundle inesperado: esperado={expected_bundle_id}, obtenido={computed_bundle_id}"
        )
    if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
        raise ValueError("El core del bundle no coincide con el fijado por este cuaderno")
    for name, expected_sha256 in _bundle_specs(manifest):
        artifact = Path(bundle_dir) / name
        if not artifact.is_file() or _sha256(artifact) != expected_sha256:
            raise ValueError(f"Artefacto ausente o inválido: {artifact}")
    return manifest

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if Path(manifest_path) != Path(bundle_dir) / "bundle_manifest.json":
        return False
    try:
        _verify_expected_bundle(bundle_dir, expected_bundle_id)
        return True
    except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _download_bundle_file(url, destination):
    destination = Path(destination)
    partial = destination.with_name(f".{destination.name}.partial")
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "ModeracionPeru-Colab-Bundle/2.0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response, partial.open("wb") as target:
            while block := response.read(1024 * 1024):
                target.write(block)
        os.replace(partial, destination)
    finally:
        if partial.exists():
            partial.unlink()

def _prepare_bundle_staging():
    staging = Path("/content/moderacion_peru_bundle_source")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    return staging

def _acquire_expected_bundle():
    staging = _prepare_bundle_staging()
    if COLAB_BUNDLE_SOURCE == "github":
        encoded_ref = urllib.parse.quote(COLAB_GITHUB_REF, safe="")
        base = (
            f"https://raw.githubusercontent.com/{COLAB_GITHUB_REPOSITORY}/"
            f"{encoded_ref}/{COLAB_GITHUB_BUNDLE_PATH}"
        )
        cache_key = urllib.parse.quote(COLAB_NOTEBOOK_BUILD_BUNDLE_ID, safe="")
        manifest_path = staging / "bundle_manifest.json"
        _download_bundle_file(f"{base}/bundle_manifest.json?bundle_id={cache_key}", manifest_path)
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            raise ValueError("El manifiesto descargado desde GitHub no es válido")
        if manifest["bundle_id"] != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError(
                "GitHub todavía no contiene el bundle fijado por este cuaderno. "
                "Sincronice resultados/colab_bundle o use COLAB_BUNDLE_SOURCE='local_upload'."
            )
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            raise RuntimeError("GitHub contiene un project_core.zip distinto al esperado")
        for name, _ in _bundle_specs(manifest):
            encoded_name = urllib.parse.quote(name, safe="")
            _download_bundle_file(f"{base}/{encoded_name}?bundle_id={cache_key}", staging / name)
    elif COLAB_BUNDLE_SOURCE == "local_upload":
        from google.colab import files

        uploaded = files.upload()
        if "bundle_manifest.json" not in uploaded:
            raise FileNotFoundError("La selección no incluyó bundle_manifest.json")
        (staging / "bundle_manifest.json").write_bytes(uploaded["bundle_manifest.json"])
        manifest = _read_manifest(staging / "bundle_manifest.json")
        if manifest.get("bundle_id") != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError("Los archivos seleccionados no pertenecen al bundle esperado")
        required = {"bundle_manifest.json", *(name for name, _ in _bundle_specs(manifest))}
        missing = sorted(required - set(uploaded))
        if missing:
            raise FileNotFoundError(f"Faltaron archivos del bundle: {missing}")
        for name in required - {"bundle_manifest.json"}:
            (staging / name).write_bytes(uploaded[name])
    else:
        raise ValueError("COLAB_BUNDLE_SOURCE debe ser 'github' o 'local_upload'")
    return staging, _verify_expected_bundle(staging)

def _write_latest_pointer(releases_dir, release_dir, manifest):
    pointer = {
        "schema_version": "1.0.0",
        "bundle_id": manifest["bundle_id"],
        "core_sha256": manifest["core"]["sha256"],
        "manifest_sha256": _sha256(Path(release_dir) / "bundle_manifest.json"),
        "published_at": datetime.now(timezone.utc).isoformat(),
    }
    latest_path = Path(releases_dir) / "latest.json"
    partial = Path(releases_dir) / f".latest-{uuid.uuid4().hex}.json"
    partial.write_text(json.dumps(pointer, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(partial, latest_path)
    return pointer

def _publish_expected_bundle(staging, releases_dir):
    manifest = _verify_expected_bundle(staging)
    releases_dir = Path(releases_dir)
    releases_dir.mkdir(parents=True, exist_ok=True)
    release_dir = releases_dir / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if release_dir.exists():
        _verify_expected_bundle(release_dir)
        release_status = "already_present_and_verified"
    else:
        partial = releases_dir / f".{COLAB_NOTEBOOK_BUILD_BUNDLE_ID}.partial-{uuid.uuid4().hex}"
        partial.mkdir()
        try:
            for name, _ in _bundle_specs(manifest):
                shutil.copyfile(Path(staging) / name, partial / name)
            shutil.copyfile(
                Path(staging) / "bundle_manifest.json",
                partial / "bundle_manifest.json",
            )
            _verify_expected_bundle(partial)
            os.replace(partial, release_dir)
        finally:
            if partial.exists():
                shutil.rmtree(partial)
        release_status = "auto_published_and_verified"
    pointer = _write_latest_pointer(releases_dir, release_dir, manifest)
    return {
        "status": release_status,
        "release_dir": release_dir,
        "latest_pointer": pointer,
    }

def _ensure_expected_drive_release(releases_dir):
    release_dir = Path(releases_dir) / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if _bundle_is_current(
        release_dir,
        release_dir / "bundle_manifest.json",
        COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
    ):
        return {"status": "already_present_and_verified", "release_dir": release_dir}
    if not COLAB_AUTO_PUBLISH_MISSING_BUNDLE:
        raise RuntimeError(
            "Drive no contiene el release esperado y COLAB_AUTO_PUBLISH_MISSING_BUNDLE=False"
        )
    staging, _ = _acquire_expected_bundle()
    return _publish_expected_bundle(staging, releases_dir)

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    RELEASES_DIR.mkdir(parents=True, exist_ok=True)
    release_check = _ensure_expected_drive_release(RELEASES_DIR)
    latest_bundle_id = COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    RELEASE_DIR = RELEASES_DIR / latest_bundle_id
    manifest = _verify_expected_bundle(RELEASE_DIR)
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    latest_pointer = _read_manifest(latest_pointer_path) if latest_pointer_path.is_file() else {}
    latest_matches_notebook = (
        latest_pointer.get("bundle_id") == COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        and latest_pointer.get("core_sha256") == COLAB_EXPECTED_CORE_SHA256
        and latest_pointer.get("manifest_sha256") == _sha256(release_manifest_path)
    )
    if latest_matches_notebook:
        release_source = (
            "auto_published_from_" + COLAB_BUNDLE_SOURCE
            if release_check["status"] == "auto_published_and_verified"
            else "latest_pointer"
        )
    else:
        # Un cuaderno reproducible puede activar su release inmutable exacto aunque
        # latest todavía apunte a otra versión; jamás mezcla código e inputs.
        release_source = "notebook_pinned_release"
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive después de "
                f"verificar o autopublicar {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'origen_del_release': release_source,
        'estado_del_release': release_check['status'],
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Prompt operacional vigente', {'ruta': OPERATIONAL_PROMPT, 'versión': '3.2.0'}, tone='success')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


estado,ya_estaba_actualizado
bundle_id,b2030f898ec03e9c7dc6b938de7453bef027c7a19b07249246595d8f4b1629cb
bundle_del_notebook_al_generarse,b2030f898ec03e9c7dc6b938de7453bef027c7a19b07249246595d8f4b1629cb
origen_del_release,latest_pointer
estado_del_release,already_present_and_verified
core_sha256,862cada62e73b099a817d15748a06cfcc19a55a004939ef7aa3cd88576a9b161
generado,2026-08-13T09:38:35.846057+00:00
versión_inmutable_drive,Ver detalle/content/drive/MyDrive/ModeracionPeru_Colab/bundle_releases/b2030f898ec03e9c7dc6b938de7453bef027c7a19b07249246595d8f4b1629cb


is_colab,Sí
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""auto"", ""device_name"": ""NVIDIA A100-SXM4-40GB"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 42405855232, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
nvidia_smi,"NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07"
cwd,/content
free_runtime_bytes,63348531200


notebook_id,03_06
run_id,03_06_working_v2_1
drive_root,/content/drive/MyDrive/ModeracionPeru_Colab
runtime_root,/content/moderacion_peru
project_root,/content/moderacion_peru/project
input_paths,"Ver detalle{ ""dataset_5_salidas"": ""/content/moderacion_peru/inputs/datos/model_ready/v2/dataset_5_salidas.jsonl"" }"
scratch_output_dir,/content/moderacion_peru/runs/03_06/03_06_working_v2_1
drive_run_dir,/content/drive/MyDrive/ModeracionPeru_Colab/runs/03_06/03_06_working_v2_1
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""cuda"", ""device_name"": ""NVIDIA A100-SXM4-40GB"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 42405855232, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
resumed,Sí


ruta,/content/moderacion_peru/project/config/prompt_operacional_ollama_v3_2.md
versión,3.2.0


## Restauración reproducible del dataset

In [2]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


status,verified_in_colab
input_key,dataset_5_salidas
path,/content/moderacion_peru/inputs/datos/model_ready/v2/dataset_5_salidas.jsonl
bytes,225478048


## Configuración y ejecución

In [3]:
from moderacion_peru.experiments import select_qwen_lora_warm_start_candidate,train_neural_experiment
DATA=COLAB_CONTEXT.input('dataset_5_salidas') if COLAB_CONTEXT else ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
OUTPUT_ROOT=COLAB_CONTEXT.scratch_output_dir if COLAB_CONTEXT else ROOT/'modelos/v2/qwen_estructurado'
DEVICE='cuda' if COLAB_CONTEXT else 'auto'
PERSISTENT_CHECKPOINT_ROOT=COLAB_CONTEXT.drive_run_dir/'trainer_checkpoints' if COLAB_CONTEXT else None
QWEN_LORA_PARENT_RUN_ID='03_05_working_v2_1'
QWEN_LORA_PARENT_MAX_LENGTH=128
STRUCTURED_PENALTIES=(0.0,0.02,0.05)
STRUCTURED_EPOCHS=1
STRUCTURED_LEARNING_RATE=2e-5
SAMPLING_SEED=20260805
TRAINING_SEED=20260805
RUN_LEGACY_FULL_TRAINING=False  # conserva el experimento histórico; no es la opción recomendada
RUN_STRUCTURED_LORA_SWEEP=True  # recomendado: tres adaptadores pequeños desde 03_05

def publish_structured_variant(label):
    if COLAB_CONTEXT is None: return
    from moderacion_peru.colab import publish_colab_outputs
    show_result(label,publish_colab_outputs(COLAB_CONTEXT),tone='success')

if RUN_LEGACY_FULL_TRAINING:
    legacy_result=run_with_progress('Qwen estructurado · full fine-tuning histórico',train_neural_experiment,DATA,OUTPUT_ROOT,experiment='qwen_structured',device=DEVICE,safe_to_damage_ratio=4.0,persistent_checkpoint_root=PERSISTENT_CHECKPOINT_ROOT,progress_unit='etapa')
    show_result('Candidato histórico independiente',legacy_result,tone='warning')

if RUN_STRUCTURED_LORA_SWEEP:
    if COLAB_CONTEXT is not None:
        from moderacion_peru.colab import restore_colab_run_outputs
        QWEN_LORA_PARENT_ROOT=COLAB_CONTEXT.runtime_root/'warm_starts'/'03_05'/QWEN_LORA_PARENT_RUN_ID
        restored_parent=restore_colab_run_outputs(COLAB_CONTEXT.drive_root,notebook_id='03_05',run_id=QWEN_LORA_PARENT_RUN_ID,destination=QWEN_LORA_PARENT_ROOT)
        show_result('Publicación 03_05 restaurada',restored_parent,tone='success')
    else:
        QWEN_LORA_PARENT_ROOT=ROOT/'modelos/v2/qwen_lora'
    qwen_lora_parent=select_qwen_lora_warm_start_candidate(QWEN_LORA_PARENT_ROOT,DATA,max_length=QWEN_LORA_PARENT_MAX_LENGTH)
    show_result('Adaptador padre verificado',{'candidate_id':qwen_lora_parent['candidate_id'],'candidate_path':qwen_lora_parent['candidate_path'],'dataset_sha256':qwen_lora_parent['dataset_sha256'],'contexto':QWEN_LORA_PARENT_MAX_LENGTH,'optimizador':'nuevo por variante'},tone='success')
    for penalty in STRUCTURED_PENALTIES:
        penalty_code=f'{int(round(penalty*100)):03d}'
        variant=f'lora03_05_structured_p{penalty_code}'
        result=run_with_progress(f'Qwen estructurado LoRA · penalización {penalty:.2f}',train_neural_experiment,DATA,OUTPUT_ROOT,experiment='qwen_structured',device=DEVICE,max_length=QWEN_LORA_PARENT_MAX_LENGTH,epochs=STRUCTURED_EPOCHS,learning_rate=STRUCTURED_LEARNING_RATE,training_seed=TRAINING_SEED,sampling_seed=SAMPLING_SEED,variant_id=variant,warm_start_candidate_path=qwen_lora_parent['candidate_path'],structured_penalty=penalty,loss_mode='weighted_bce',safe_to_damage_ratio=4.0,persistent_checkpoint_root=PERSISTENT_CHECKPOINT_ROOT,progress_unit='etapa')
        show_result(f'Candidato {variant}',result,tone='success')
        publish_structured_variant(f'Publicación {variant}')

if not (RUN_LEGACY_FULL_TRAINING or RUN_STRUCTURED_LORA_SWEEP):
    show_summary('Qwen estructurado de bajo costo preparado',{'recomendado':'LoRA entrenable restaurado desde el candidato 03_05','comparación':'penalización 0, 0.02 y 0.05 sobre las mismas filas','épocas':STRUCTURED_EPOCHS,'learning_rate':STRUCTURED_LEARNING_RATE,'semillas':f'sampling={SAMPLING_SEED}; training={TRAINING_SEED}','estado_optimizador':'nuevo en cada candidato','candidato_full_histórico':'se conserva; no se sobrescribe','test':'natural completo, sellado'},tone='neutral')
if COLAB_CONTEXT is not None and (RUN_LEGACY_FULL_TRAINING or RUN_STRUCTURED_LORA_SWEEP):
    from moderacion_peru.colab import publish_colab_outputs
    final_publication=publish_colab_outputs(COLAB_CONTEXT)
    show_result('Publicación final automática',final_publication,tone='success')

status,restored_and_sha256_verified
source,/content/drive/MyDrive/ModeracionPeru_Colab/runs/03_05/03_05_working_v2_1
destination,/content/moderacion_peru/warm_starts/03_05/03_05_working_v2_1


candidate_id,qwen_lora-4aa5ce04df05
candidate_path,/content/moderacion_peru/warm_starts/03_05/03_05_working_v2_1/runs/qwen_lora-4aa5ce04df057144/candidate.json
dataset_sha256,013d60ba1b173d7752f453d5d05629a3439b09c71f0c343da1b5e498662c1f86
contexto,128
optimizador,nuevo por variante


Qwen estructurado LoRA · penalización 0.00: 0etapa [00:00, ?etapa/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro Auprc Damage,Macro Auprc Five
1,0.131400,0.345260,0.538332,0.625019


...

schema_version,1.2.0
published_at,2026-08-13T12:51:58.218392+00:00
notebook_id,03_06
run_id,03_06_working_v2_1
taxonomy_contract,moderacion_peru_5_salidas_v2
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""cuda"", ""device_name"": ""NVIDIA A100-SXM4-40GB"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 42405855232, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
publication_slot,a
included_files,59
excluded_trainer_checkpoint_files,17
trainer_checkpoint_storage,trainer_checkpoints
archive,"Ver detalle{ ""name"": ""publications/run_outputs-a.tar"", ""sha256"": ""0579b813fc6786755fcb24c3a42f014ea5c0ed9b43ff3839fad6941f4cbed480"", ""bytes"": 2488483840, ""format"": ""tar_uncompressed"", ""verification"": ""full_readback_after_close"" }"


Qwen estructurado LoRA · penalización 0.05: 0etapa [00:00, ?etapa/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Publicación o checkpoint en Drive

Cada época completa se guarda y verifica por separado en `trainer_checkpoints`. Al terminar una corrida 03_x se publica automáticamente el candidato final en una de dos ranuras redundantes. Esta celda permite repetir manualmente esa publicación; no vuelve a incluir los directorios transitorios de `Trainer`.

In [ ]:
PUBLISH_TO_DRIVE = True
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None and globals().get('AUTO_PUBLISH_CHECKPOINTS'):
    show_callout('Checkpoint automático activo', 'La recuperación, los checkpoints periódicos, Ctrl+C y cada cierre de campaña ya publican una copia verificable en Drive.', tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación manual desactivada', 'Los entrenamientos 03_x ya publican automáticamente al completar; active esta celda solo para repetir la publicación final.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] A. Yang, A. Li, B. Yang, et al., "Qwen3 Technical Report," arXiv:2505.09388, 2025, doi: 10.48550/arXiv.2505.09388.

[2] Qwen Team, "Model Card: Qwen/Qwen3-0.6B-Base," Hugging Face Hub, revision da87bfb608c14b7cf20ba1ce41287e8de496c0cd, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-0.6B-Base/tree/da87bfb608c14b7cf20ba1ce41287e8de496c0cd

[3] C. N. Silla Jr. and A. A. Freitas, "A Survey of Hierarchical Classification Across Different Application Domains," Data Min. Knowl. Discovery, vol. 22, no. 1–2, pp. 31–72, 2011, doi: 10.1007/s10618-010-0175-9.

[4] G. Tsoumakas and I. Katakis, "Multi-Label Classification: An Overview," Int. J. Data Warehousing Mining, vol. 3, no. 3, pp. 1–13, 2007, doi: 10.4018/jdwm.2007070101.

[5] E. J. Hu, Y. Shen, P. Wallis, et al., "LoRA: Low-Rank Adaptation of Large Language Models," in Proc. ICLR, 2022. [Online]. Available: https://openreview.net/forum?id=nZeVKeeFYf9

[6] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[7] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.